### Import Libraries

In [ ]:
from langchain_cohere import ChatCohere
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_cohere import CohereEmbeddings
from langchain_community.document_loaders import PyPDFLoader

# Initializa LLM

In [ ]:
import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

from langchain_community.embeddings import OCIGenAIEmbeddings
embed_model = OCIGenAIEmbeddings(
    model_id=properties.getEmbeddingModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
    auth_type='INSTANCE_PRINCIPAL',)


from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI


llm = ChatOCIGenAI(
      model_id='cohere.command-r-08-2024',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)    

### Load PDF

In [ ]:

loader = PyPDFLoader("./finance_data.pdf")
pages = loader.load()


### Combined raw text from all pages

In [ ]:
raw_text = ''

for i, doc in enumerate(pages):
    text = doc.page_content
    if text:
        raw_text += text

### Text splitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100    
)

texts = text_splitter.split_text(raw_text)

### Text chunk into a document structure

In [ ]:

from langchain_core.documents import Document

docs = []
for i in range(len(texts)):
    doc = Document(page_content=texts[i])
    docs.append(doc)

### Initialize a vector database

In [ ]:

vectordb = Chroma(
    collection_name='summaries',
    embedding_function=embed_model,
    persist_directory='./data'  
)

vectordb.add_documents(docs)

### Create a retriever

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k": 4})


### Define a prompt template

In [ ]:

PRODUCT_BOT_PROMPT = """
    You are a smart assistant.
    Your response must only be in English.
    Ensure your answers are relevant to the query with reference to provided context and not outside the context.
    Your responses should be elaborate and up to the mark referring to the context only.
    Do not include the keyword context in the final answer
    
    CONTEXT:
    {context}

    QUESTION: {question}

    YOUR ANSWER:
"""


from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(PRODUCT_BOT_PROMPT)


### Chaining inputs/outputs

In [ ]:

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# Define the full processing pipeline 
# 1. Takes query input
# 2. Retrieves relevant context from vector DB
# 3. Fills in the prompt template
# 4. Sends it to the LLM
# 5. Parses the output string

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### Process the query

In [ ]:

""" Example Questions to try
Who is Mr. Raza
Who is Steve Jobs
What is Form 10 - K
Explain acquisitions
List different events from year 2020 to 2023
"""

query="List different events from year 2020 to 2023"

result = chain.invoke(query)

# Print response 
print("Response:", result)